In [8]:
# Cell 1: 环境
import os, sys, json
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from config import get_driver
from src.ci.patent_service import create_patent_family, list_patent_families_for_organization
from src.ci.clinical_trial_service import create_clinical_trial, list_trials_for_program
driver = get_driver()
print("✅ 模块导入成功")

✅ 模块导入成功


In [9]:
# Cell 2: 创建专利族 + 临床试验
# 1. 找现有组织、项目、构建体
with driver.session() as s:
    org = s.run("""
        MATCH (o:Organization)
        WHERE EXISTS { MATCH (o)-[:DEVELOPS]->(:DevelopmentProgram) }
        RETURN o.organization_id AS id, o.canonical_name AS name
        ORDER BY o.canonical_name LIMIT 1
    """).single()
    if not org:
        raise RuntimeError("数据库里没有带项目的组织，请先跑 05 notebook")
    print(f"使用组织: {org['name']} ({org['id']})")

with driver.session() as s:
    prog = s.run("""
        MATCH (d:DevelopmentProgram)
        RETURN d.program_id AS id, d.canonical_name AS name
        ORDER BY d.canonical_name LIMIT 1
    """).single()
    print(f"默认项目: {prog['name']} ({prog['id']})")

# ★ 关键改动：找出所有"有构建体关联"的项目，专利族/临床试验将挂到这些项目上
with driver.session() as s:
    progs_with_constructs = [
        r["pid"] for r in s.run("""
            MATCH (ec:EngineeredPhageConstruct)-[:ASSOCIATED_WITH]->(dp:DevelopmentProgram)
            RETURN DISTINCT dp.program_id AS pid
        """)
    ]

if not progs_with_constructs:
    print("⚠️ 未发现有构建体关联的项目，将回退到默认项目")
    progs_with_constructs = [prog["id"]]
else:
    print(f"✅ 有构建体关联的项目：{progs_with_constructs}")

# 2. 创建 3 个专利族
patent_ids = []
patent_specs = [
    dict(
        title="Engineered bacteriophage with expanded host range",
        priority_date="2022-03-15",
        publication_numbers=["WO2023/123456A1", "US2024/0123456A1"],
        jurisdictions=["WO", "US"],
        applicants=["Example Biotech Ltd."],
        assignees=["Example Biotech Ltd."],
        legal_status="granted",
        claim_scope_summary="权利要求 1 覆盖尾纤维改造以扩展对 KL47 型 K. pneumoniae 的宿主范围。",
        source_url="https://patents.example.com/WO2023123456",
    ),
    dict(
        title="Phage-encoded biofilm degrading enzyme",
        priority_date="2023-06-01",
        publication_numbers=["WO2024/654321A1"],
        jurisdictions=["WO"],
        applicants=["Example Biotech Ltd."],
        assignees=["Example Biotech Ltd."],
        legal_status="pending",
        claim_scope_summary="权利要求 1 覆盖在噬菌体基因组中插入生物膜降解酶编码序列。",
        source_url="https://patents.example.com/WO2024654321",
    ),
    dict(
        title="Anti-CRISPR payload for phage therapy",
        priority_date="2023-11-10",
        publication_numbers=["US2025/0011223A1"],
        jurisdictions=["US"],
        applicants=["Example Biotech Ltd."],
        assignees=["Example Biotech Ltd."],
        legal_status="pending",
        claim_scope_summary="权利要求 1 覆盖装载 anti-CRISPR 蛋白的工程化噬菌体。",
        source_url="https://patents.example.com/US20250011223",
    ),
]
for spec in patent_specs:
    pfid = create_patent_family(
        driver,
        title=spec["title"],
        priority_date=spec["priority_date"],
        publication_numbers=spec["publication_numbers"],
        jurisdictions=spec["jurisdictions"],
        applicants=spec["applicants"],
        assignees=spec["assignees"],
        legal_status=spec["legal_status"],
        claim_scope_summary=spec["claim_scope_summary"],
        source_url=spec["source_url"],
        organization_id=org["id"],
        program_ids=progs_with_constructs,   # ★ 挂到有构建体的所有项目
        actor_id="demo",
    )
    patent_ids.append(pfid)
    print(f"✅ 专利族: {pfid}")

# 3. 创建 2 个临床试验
trial_ids = []
trial_specs = [
    dict(
        title="PhageBank-CF Phase 1/2 in cystic fibrosis P. aeruginosa",
        nct_id="NCT05012345",
        phase="phase_1_2",
        status="completed",
        indication="Cystic fibrosis lung infection",
        sponsor="Adaptive Phage Therapeutics",
        start_date="2021-06-01",
        completion_date="2023-12-31",
        enrollment=36,
        primary_endpoint="Safety and tolerability",
    ),
    dict(
        title="BX211 Phase 2 in diabetic foot osteomyelitis",
        nct_id="NCT06098765",
        phase="phase_2",
        status="active",
        indication="Diabetic foot osteomyelitis",
        sponsor="BiomX",
        start_date="2024-02-01",
        completion_date="2026-06-30",
        enrollment=120,
        primary_endpoint="Clinical response at week 12",
    ),
]
for spec in trial_specs:
    # ★ 每个试验挂到第一个"有构建体的项目"上，避免和专利族一样错位
    tid = create_clinical_trial(
        driver,
        title=spec["title"],
        nct_id=spec["nct_id"],
        phase=spec["phase"],
        status=spec["status"],
        indication=spec["indication"],
        sponsor=spec["sponsor"],
        organization_id=org["id"],
        program_id=progs_with_constructs[0],
        start_date=spec["start_date"],
        completion_date=spec["completion_date"],
        enrollment=spec["enrollment"],
        primary_endpoint=spec["primary_endpoint"],
        actor_id="demo",
    )
    trial_ids.append(tid)
    print(f"✅ 临床试验: {tid}")

print(f"\n共创建 {len(patent_ids)} 个专利族、{len(trial_ids)} 个临床试验")

使用组织: Adaptive Phage Therapeutics (CI:ORG:55DF2E8C)
默认项目: Athinia (CI:PROG:A55770C2)
⚠️ 未发现有构建体关联的项目，将回退到默认项目
✅ 专利族: CI:PATENT:BD49B1B3
✅ 专利族: CI:PATENT:DD4AD008
✅ 专利族: CI:PATENT:B8C0BEE0
✅ 临床试验: CI:TRIAL:E1843F88
✅ 临床试验: CI:TRIAL:1A53E4F0

共创建 3 个专利族、2 个临床试验


In [10]:
# Cell 2.5: 一次性修复关系 —— 把孤立构建体、专利族、临床试验对齐到同一个项目
with driver.session() as s:
    # 1. 选目标项目：优先挑已经有专利族的项目
    target = s.run("""
        MATCH (dp:DevelopmentProgram)-[:COVERED_BY]->(:PatentFamily)
        RETURN dp.program_id AS pid, dp.canonical_name AS name
        ORDER BY dp.canonical_name LIMIT 1
    """).single()
    if not target:
        target = s.run("""
            MATCH (dp:DevelopmentProgram)
            RETURN dp.program_id AS pid, dp.canonical_name AS name
            ORDER BY dp.canonical_name LIMIT 1
        """).single()
    if not target:
        raise RuntimeError("数据库里没有项目，请先跑 05 notebook")

    target_pid = target["pid"]
    print(f"目标项目: {target['name']} ({target_pid})\n")

    # 2. 补齐孤立构建体 -> ASSOCIATED_WITH -> 目标项目
    orphans = s.run("""
        MATCH (ec:EngineeredPhageConstruct)
        WHERE NOT EXISTS { MATCH (ec)-[:ASSOCIATED_WITH]->(:DevelopmentProgram) }
        RETURN ec.construct_id AS cid, ec.public_name AS name
    """).data()
    if orphans:
        print(f"孤立构建体 {len(orphans)} 个，正在挂到项目...")
        for o in orphans:
            s.run("""
                MATCH (ec:EngineeredPhageConstruct {construct_id: $cid})
                MATCH (dp:DevelopmentProgram {program_id: $pid})
                MERGE (ec)-[:ASSOCIATED_WITH]->(dp)
            """, cid=o["cid"], pid=target_pid)
            print(f"  ✅ {o['name']} ({o['cid']})")
    else:
        print("✅ 所有构建体已关联到项目")

    # 3. 补齐未关联项目的专利族
    patents = s.run("""
        MATCH (pf:PatentFamily)
        WHERE NOT EXISTS { MATCH (:DevelopmentProgram)-[:COVERED_BY]->(pf) }
        RETURN pf.patent_family_id AS pfid, pf.title AS title
    """).data()
    if patents:
        print(f"\n未关联项目的专利族 {len(patents)} 个，正在挂到项目...")
        for p in patents:
            s.run("""
                MATCH (pf:PatentFamily {patent_family_id: $pfid})
                MATCH (dp:DevelopmentProgram {program_id: $pid})
                MERGE (dp)-[:COVERED_BY]->(pf)
            """, pfid=p["pfid"], pid=target_pid)
            print(f"  ✅ {p['title'][:50]}")
    else:
        print("✅ 所有专利族已关联到项目")

    # 4. 补齐未关联项目的临床试验
    trials = s.run("""
        MATCH (ct:ClinicalTrial)
        WHERE NOT EXISTS { MATCH (:DevelopmentProgram)-[:HAS_TRIAL]->(ct) }
        RETURN ct.trial_id AS tid, ct.nct_id AS nct
    """).data()
    if trials:
        print(f"\n未关联项目的临床试验 {len(trials)} 个，正在挂到项目...")
        for t in trials:
            s.run("""
                MATCH (ct:ClinicalTrial {trial_id: $tid})
                MATCH (dp:DevelopmentProgram {program_id: $pid})
                MERGE (dp)-[:HAS_TRIAL]->(ct)
            """, tid=t["tid"], pid=target_pid)
            print(f"  ✅ {t['nct']}")
    else:
        print("✅ 所有临床试验已关联到项目")

    # 5. 最终校验
    print("\n=== 最终校验 ===")
    print("--- biofilm_disruption 构建体 -> 项目 ---")
    for row in s.run("""
        MATCH (es:EngineeringStrategy {strategy_type: 'biofilm_disruption'})
              <-[:IMPLEMENTS]-(ec:EngineeredPhageConstruct)
        OPTIONAL MATCH (ec)-[:ASSOCIATED_WITH]->(dp:DevelopmentProgram)
        RETURN ec.public_name AS c, dp.program_id AS pid, dp.canonical_name AS p
    """):
        print(f"  {row['c']} -> {row['pid']} ({row['p']})")

    print("--- 专利族 -> 项目 ---")
    for row in s.run("""
        MATCH (pf:PatentFamily)<-[:COVERED_BY]-(dp:DevelopmentProgram)
        RETURN pf.patent_family_id AS pf, pf.title AS t, dp.program_id AS pid
        ORDER BY pf.title
    """):
        print(f"  {row['pf']}  {row['t'][:45]} -> {row['pid']}")

目标项目: Athinia (CI:PROG:A55770C2)

孤立构建体 10 个，正在挂到项目...
  ✅ vB_Kpn_HRE_001 (ENG:CONST:EAE7F653)
  ✅ vB_Pae_HRE_002 (ENG:CONST:2EB75660)
  ✅ vB_Kpn_TFE_003 (ENG:CONST:0598E778)
  ✅ vB_Eco_TFE_004 (ENG:CONST:D260CE8B)
  ✅ vB_Pae_LYE_005 (ENG:CONST:6EA9D6DA)
  ✅ vB_Sau_LYE_006 (ENG:CONST:DE1A3C78)
  ✅ vB_Pae_BFD_007 (ENG:CONST:2B0A0785)
  ✅ vB_Sau_BFD_008 (ENG:CONST:E2ED2755)
  ✅ vB_Kpn_PLD_009 (ENG:CONST:798DE2FF)
  ✅ vB_Pae_PLD_010 (ENG:CONST:F9FE6212)
✅ 所有专利族已关联到项目
✅ 所有临床试验已关联到项目

=== 最终校验 ===
--- biofilm_disruption 构建体 -> 项目 ---
  vB_Sau_BFD_008 -> CI:PROG:A55770C2 (Athinia)
  vB_Pae_BFD_007 -> CI:PROG:A55770C2 (Athinia)
--- 专利族 -> 项目 ---
  CI:PATENT:B8C0BEE0  Anti-CRISPR payload for phage therapy -> CI:PROG:A55770C2
  CI:PATENT:BD49B1B3  Engineered bacteriophage with expanded host r -> CI:PROG:A55770C2
  CI:PATENT:DD4AD008  Phage-encoded biofilm degrading enzyme -> CI:PROG:A55770C2


In [11]:
# Cell 3: 校验
print("=== 组织拥有的专利族 ===")
for r in list_patent_families_for_organization(driver, org["id"]):
    print(f"  {r['id']}  {r['title'][:50]}  [{r['legal_status']}]")

print("\n=== 项目的临床试验 ===")
for r in list_trials_for_program(driver, prog["id"]):
    print(f"  {r['id']}  {r['nct_id']}  {r['phase']}  {r['status']}  {r['title'][:50]}")

=== 组织拥有的专利族 ===
  CI:PATENT:B8C0BEE0  Anti-CRISPR payload for phage therapy  [pending]
  CI:PATENT:DD4AD008  Phage-encoded biofilm degrading enzyme  [pending]
  CI:PATENT:BD49B1B3  Engineered bacteriophage with expanded host range  [granted]

=== 项目的临床试验 ===
  CI:TRIAL:1A53E4F0  NCT06098765  phase_2  active  BX211 Phase 2 in diabetic foot osteomyelitis
  CI:TRIAL:E1843F88  NCT05012345  phase_1_2  completed  PhageBank-CF Phase 1/2 in cystic fibrosis P. aerug


In [12]:
# Cell 4: 技术简报现在能看到专利族了
from src.engineering_intelligence.strategy_classifier import get_all_strategies
from src.engineering_intelligence.technology_brief import generate_technology_brief

strategies = get_all_strategies(driver)
if strategies:
    brief = generate_technology_brief(
        driver,
        strategy_id=strategies[0]["id"],
        persist=True,
    )
    print(f"brief_id: {brief['brief_id']}")
    print(f"patent_families: {len(brief['patent_families'])} 条")
    for pf in brief["patent_families"]:
        print(f"  - {pf['patent_family_id']}  {pf['title'][:50]}  [{pf['legal_status']}]")
    print(f"\nrecommended_next_steps:")
    for s in brief["recommended_next_steps"]:
        print(f"  - {s}")

brief_id: CI:BRIEF:8DCF54F5
patent_families: 3 条
  - CI:PATENT:B8C0BEE0  Anti-CRISPR payload for phage therapy  [pending]
  - CI:PATENT:DD4AD008  Phage-encoded biofilm degrading enzyme  [pending]
  - CI:PATENT:BD49B1B3  Engineered bacteriophage with expanded host range  [granted]

recommended_next_steps:
  - 当前证据成熟度均停留在体外阶段，临床转化存在明显不确定性。


In [13]:
# Cell 5: 生成 Competitor Brief，检查 patent_landscape + clinical_trials
import importlib
import src.ci.competitor_profile as cp
import src.ci.competitor_brief as cb
importlib.reload(cp)
importlib.reload(cb)
from src.ci.competitor_brief import generate_competitor_brief

brief = generate_competitor_brief(
    driver,
    organization_id=org["id"],
    days_back=730,
    persist=True,
)

print(f"brief_id: {brief.get('brief_id')}")

pl = brief.get("patent_landscape", {})
print(f"\n=== Patent Landscape ===")
print(f"  总数: {pl.get('total_families')}")
print(f"  按状态: {pl.get('by_legal_status')}")
print(f"  按法域: {pl.get('by_jurisdiction')}")
for pf in pl.get("recent_families", []):
    print(f"  - {pf['patent_family_id']}  {pf['title'][:50]}  [{pf['legal_status']}]")

ct = brief.get("clinical_trials", {})
print(f"\n=== Clinical Trials ===")
print(f"  总数: {ct.get('total_trials')}")
print(f"  按阶段: {ct.get('by_phase')}")
print(f"  按状态: {ct.get('by_status')}")
for t in ct.get("active_trials", []):
    print(f"  - {t['nct_id']}  {t['phase']}  {t['status']}  {t['title'][:50]}")

print(f"\n=== Recommended Next Steps ===")
for s in brief.get("recommended_next_steps", []):
    print(f"  - {s}")

brief_id: CI:BRIEF:F8B1ED66

=== Patent Landscape ===
  总数: 3
  按状态: {'pending': 2, 'granted': 1}
  按法域: {'US': 2, 'WO': 2}
  - CI:PATENT:B8C0BEE0  Anti-CRISPR payload for phage therapy  [pending]
  - CI:PATENT:DD4AD008  Phage-encoded biofilm degrading enzyme  [pending]
  - CI:PATENT:BD49B1B3  Engineered bacteriophage with expanded host range  [granted]

=== Clinical Trials ===
  总数: 2
  按阶段: {'phase_2': 1, 'phase_1_2': 1}
  按状态: {'active': 1, 'completed': 1}
  - NCT06098765  phase_2  active  BX211 Phase 2 in diabetic foot osteomyelitis

=== Recommended Next Steps ===
  - 建议跟进近期重大事件，评估对内部战略的影响
  - 对手拥有 3 个专利族，建议启动 FTO 初筛
  - 对手有 1 项活跃临床试验，建议持续跟踪
  - 补充数据缺口，完善竞争对手档案


In [14]:
# Cell: IntelligenceQuestion + WatchlistItem demo
import importlib
import src.decision_support.intelligence_question as iq_mod
import src.decision_support.watchlist_service as wl_mod
importlib.reload(iq_mod)
importlib.reload(wl_mod)

from src.decision_support.intelligence_question import (
    create_intelligence_question, list_intelligence_questions, link_brief_to_question,
)
from src.decision_support.watchlist_service import (
    create_watchlist_item, list_watchlist_items,
)

# 1. 创建 2 个情报问题
iq1 = create_intelligence_question(
    driver,
    question_text="哪些竞争对手正在布局工程化噬菌体治疗 CRKP？",
    question_type="market",
    requester_id="VP_Strategy",
    priority="high",
    actor_id="demo",
)
print(f"✅ 情报问题 1: {iq1}")

iq2 = create_intelligence_question(
    driver,
    question_text="尾纤维工程在扩大宿主范围方面有哪些公开证据？",
    question_type="technology",
    requester_id="R&D_Lead",
    priority="medium",
    actor_id="demo",
)
print(f"✅ 情报问题 2: {iq2}")

# 2. 创建 3 个关注项
wi1 = create_watchlist_item(
    driver,
    target_type="organization",
    target_id=org["id"],
    reason="持有 3 个工程化噬菌体专利族，且有 2 项活跃临床试验",
    priority="high",
    owner_id="VP_Strategy",
    review_date="2027-01-01",
    derived_from_brief_id=brief["brief_id"],
    actor_id="demo",
)
print(f"✅ 关注项 1（组织）: {wi1}")

wi2 = create_watchlist_item(
    driver,
    target_type="engineering_strategy",
    target_id=strategies[0]["id"],
    reason="生物膜破坏策略公开证据增长，需评估内部能力差距",
    priority="medium",
    owner_id="R&D_Lead",
    review_date="2027-03-01",
    actor_id="demo",
)
print(f"✅ 关注项 2（策略）: {wi2}")

wi3 = create_watchlist_item(
    driver,
    target_type="patent_family",
    target_id=patent_ids[0],
    reason="核心专利族 granted，需启动 FTO 初筛",
    priority="high",
    owner_id="IP_Lead",
    review_date="2027-02-01",
    actor_id="demo",
)
print(f"✅ 关注项 3（专利族）: {wi3}")

# 3. 把简报关联到问题
link_brief_to_question(driver, brief["brief_id"], iq1, actor_id="demo")
print(f"\n✅ 简报 {brief['brief_id']} 已关联到问题 {iq1}")

# 4. 列出
print("\n=== 情报问题列表 ===")
for q in list_intelligence_questions(driver):
    print(f"  [{q['status']}] {q['question_id']}  [{q['question_type']}]  {q['question_text'][:50]}  简报数={q['brief_count']}")

print("\n=== 关注项列表 ===")
for w in list_watchlist_items(driver):
    print(f"  [{w['priority']}] {w['watchlist_item_id']}  {w['target_type']} → {w['target_name']}")
    print(f"     理由: {w['reason'][:60]}")
    print(f"     复核: {w['review_date']}")

✅ 情报问题 1: CI:IQ:1638A536
✅ 情报问题 2: CI:IQ:D85F392D
✅ 关注项 1（组织）: CI:WATCH:D91B5E03
✅ 关注项 2（策略）: CI:WATCH:9BE21879
✅ 关注项 3（专利族）: CI:WATCH:87EF0ED3

✅ 简报 CI:BRIEF:F8B1ED66 已关联到问题 CI:IQ:1638A536

=== 情报问题列表 ===
  [open] CI:IQ:D85F392D  [technology]  尾纤维工程在扩大宿主范围方面有哪些公开证据？  简报数=0
  [open] CI:IQ:1638A536  [market]  哪些竞争对手正在布局工程化噬菌体治疗 CRKP？  简报数=1

=== 关注项列表 ===
  [medium] CI:WATCH:9BE21879  engineering_strategy → biofilm_disruption
     理由: 生物膜破坏策略公开证据增长，需评估内部能力差距
     复核: 2027-03-01
  [high] CI:WATCH:87EF0ED3  patent_family → Engineered bacteriophage with expanded host range
     理由: 核心专利族 granted，需启动 FTO 初筛
     复核: 2027-02-01
  [high] CI:WATCH:D91B5E03  organization → Adaptive Phage Therapeutics
     理由: 持有 3 个工程化噬菌体专利族，且有 2 项活跃临床试验
     复核: 2027-01-01
